## P1

In [11]:
"""
P1 : carte des régimes dans l'espace des coûts.
Usage : python analyse_P1.py P1.csv
Une cellule est en "coexistence" si, dans au moins la moitié des runs,
coopérateurs et compétiteurs coexistent pendant au moins SEUIL ticks.
Sinon, elle prend l'issue finale majoritaire (coopération ou compétition).
"""
import sys
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

CSV = "./resultats/P1.csv"
REF = (0.8, 0.8)      # point de référence (c_h, c_d)
SEUIL = 500           # durée minimale de coexistence (ticks) = 1/4 de l'horizon

with open(CSV, encoding="utf-8") as f:
    skip = next(i for i, line in enumerate(f) if line.startswith('"[run number]"'))
df = pd.read_csv(CSV, skiprows=skip)
dove = df["dove-fixation?"].astype(str).str.lower() == "true"
df["issue"] = 1                      # 0 = coopération, 1 = compétition
df.loc[dove, "issue"] = 0

cellules = df.groupby(["c-h-slider", "c-d-slider"])
regime = cellules["issue"].agg(lambda s: s.value_counts().idxmax())
regime[cellules["[step]"].median() >= SEUIL] = 2      # 2 = coexistence
counts = regime.value_counts()

grid = regime.unstack("c-h-slider")
ch, cd = grid.columns.values, grid.index.values
pas = ch[1] - ch[0]
couleurs = ["#edf8b1", "#7fcdbb", "#2c7fb8"]

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(grid.values, origin="lower", cmap=ListedColormap(couleurs), vmin=0, vmax=2,
          extent=[ch[0] - pas/2, ch[-1] + pas/2, cd[0] - pas/2, cd[-1] + pas/2], aspect="auto")
ax.plot([0, 1], [0, 1], "--", color="0.35", lw=1)
ax.text(0.3, 0.35, "$c_h = c_d$", rotation=37, fontsize=9, color="0.35")
ax.plot(*REF, marker="*", markersize=16, color="white", markeredgecolor="black", zorder=3)
ax.annotate("Reference", xy=REF, xytext=(REF[0] - 0.35, REF[1] + 0.1),
            fontsize=9, arrowprops=dict(arrowstyle="->", color="black"))
ax.legend(handles=[Patch(color=couleurs[2], label=f"Coexistence ({counts.get(2, 0)})"),
                   Patch(color=couleurs[0], label=f"Coopération ({counts.get(0, 0)})"),
                   Patch(color=couleurs[1], label=f"Compétition ({counts.get(1, 0)})")],
          loc="lower right", fontsize=9)
ax.set_xlabel("Competition cost  $c_h$")
ax.set_ylabel("Cooperation cost  $c_d$")
ax.set_title("P1 — States according costs structure\n"
             "(simulation ; Z=100, N=5, T=0.4, $x_0$=0.5, R=1 ; 50 replications)", fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig("./resultats/p1.png", dpi=300)
print(counts.rename({0: "cooperation", 1: "competition", 2: "coexistence"}).to_string())

issue
cooperation    252
competition    168
coexistence     21


## P2

In [19]:
"""
P2 : effet du niveau de ressource R.
Usage : python analyse_P2.py P2.csv

Chaque run est classé selon la même logique que P1 :
- coexistence  : coopérateurs et compétiteurs coexistent au moins SEUIL ticks
- coopération  : fixation en coopération avant SEUIL ticks
- compétition  : fixation en compétition avant SEUIL ticks
"""
import sys
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV = "./resultats/P2.csv"
SEUIL = 500           # même seuil que P1 (1/4 de l'horizon)
R_REF = 1             # valeur de référence

with open(CSV, encoding="utf-8") as f:
    skip = next(i for i, line in enumerate(f) if line.startswith('"[run number]"'))
df = pd.read_csv(CSV, skiprows=skip)
df = df[df["R-slider"] > 0]      # R = 0 : plus de ressource, cas dégénéré exclu

dove = df["dove-fixation?"].astype(str).str.lower() == "true"
df["regime"] = "Competition"
df.loc[dove, "regime"] = "Cooperation"
df.loc[df["[step]"] >= SEUIL, "regime"] = "Coexistence"

# Part des runs dans chaque régime, pour chaque valeur de R
parts = (pd.crosstab(df["R-slider"], df["regime"], normalize="index")
           .reindex(columns=["Competition", "Coexistence", "Cooperation"], fill_value=0) * 100)
print(parts.round(0).astype(int).to_string())

# Figure : une courbe par régime
couleurs = {"Cooperation": "#e6a817", "Coexistence": "#2c7fb8", "Competition": "#3a9e8f"}
fig, ax = plt.subplots(figsize=(8, 4.5))
for regime, couleur in couleurs.items():
    ax.plot(parts.index, parts[regime], "o-", color=couleur, lw=2, ms=5, label=regime)
ax.axvline(R_REF, ls=":", color="0.4", lw=1)
ax.text(R_REF + 0.02, 95, "reference", fontsize=8, color="0.4")
ax.set_ylim(-3, 103)
ax.set_xlabel("Resource level  $R$")
ax.set_ylabel("Share of runs (%)")
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
n = df.groupby("R-slider").size().iloc[0]
ax.set_title("P2 — Regimes by resource level\n"
             f"(simulation ; Z=100, N=5, $c_h$=$c_d$=0.8, T=0.4, $x_0$=0.5 ; {n} replications)",
             fontsize=10)
fig.tight_layout()
fig.savefig("./resultats/p2.png", dpi=300)

# Chiffres pour le texte
horizon = df["[step]"].max()
fin = df[df["[step]"] >= horizon].groupby("R-slider")["prop-dove"].median()
print("\nPart médiane de coopérateurs dans les runs encore en coexistence à l'horizon :")
print((fin * 100).round(0).to_string())
print("\nDurée médiane avant fixation (ticks) :")
print(df.groupby("R-slider")["[step]"].median().to_string())

regime    Competition  Coexistence  Cooperation
R-slider                                       
0.1                20           80            0
0.2                10           90            0
0.3                22           78            0
0.4                26           74            0
0.5                24           76            0
0.6                32           68            0
0.7                22           78            0
0.8                22           78            0
0.9                30           66            4
1.0                36           62            2
1.1                32           56           12
1.2                26           48           26
1.3                18           36           46
1.4                12           16           72
1.5                 0            4           96
1.6                 0            0          100
1.7                 2            0           98
1.8                 2            0           98
1.9                 0            0      

## P3

In [20]:
"""
P3 : effet du seuil d'engagement T.
Usage : python analyse_P3.py P3.csv

Chaque run est classé selon la même logique que P1 :
- coexistence  : coopérateurs et compétiteurs coexistent au moins SEUIL ticks
- coopération  : fixation en coopération avant SEUIL ticks
- compétition  : fixation en compétition avant SEUIL ticks
"""
import sys
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV = "./resultats/P3.csv"
SEUIL = 500           # même seuil que P1 (1/4 de l'horizon)
T_REF = 0.4           # valeur de référence

with open(CSV, encoding="utf-8") as f:
    skip = next(i for i, line in enumerate(f) if line.startswith('"[run number]"'))
df = pd.read_csv(CSV, skiprows=skip)
df = df[df["T-slider"] > 0]      # T = 0 : cas dégénéré exclu

dove = df["dove-fixation?"].astype(str).str.lower() == "true"
df["regime"] = "Competition"
df.loc[dove, "regime"] = "Cooperation"
df.loc[df["[step]"] >= SEUIL, "regime"] = "Coexistence"

# Part des runs dans chaque régime, pour chaque valeur de R
parts = (pd.crosstab(df["T-slider"], df["regime"], normalize="index")
           .reindex(columns=["Competition", "Coexistence", "Cooperation"], fill_value=0) * 100)
print(parts.round(0).astype(int).to_string())

# Figure : une courbe par régime
couleurs = {"Cooperation": "#e6a817", "Coexistence": "#2c7fb8", "Competition": "#3a9e8f"}
fig, ax = plt.subplots(figsize=(8, 4.5))
for regime, couleur in couleurs.items():
    ax.plot(parts.index, parts[regime], "o-", color=couleur, lw=2, ms=5, label=regime)
ax.axvline(T_REF, ls=":", color="0.4", lw=1)
ax.text(T_REF + 0.01, 95, "reference", fontsize=8, color="0.4")
ax.set_ylim(-3, 103)
ax.set_xlabel("Engagement threshold  $T$")
ax.set_ylabel("Share of runs (%)")
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
n = df.groupby("T-slider").size().iloc[0]
ax.set_title("P3 — Regimes by engagement threshold\n"
             f"(simulation ; Z=100, N=5, $c_h$=$c_d$=0.8, R=1, $x_0$=0.5 ; {n} replications)",
             fontsize=10)
fig.tight_layout()
fig.savefig("./resultats/p3.png", dpi=300)

# Chiffres pour le texte
horizon = df["[step]"].max()
fin = df[df["[step]"] >= horizon].groupby("T-slider")["prop-dove"].median()
print("\nPart médiane de coopérateurs dans les runs encore en coexistence à l'horizon :")
print((fin * 100).round(0).to_string())
print("\nDurée médiane avant fixation (ticks) :")
print(df.groupby("T-slider")["[step]"].median().to_string())

regime    Competition  Coexistence  Cooperation
T-slider                                       
0.1               100            0            0
0.2               100            0            0
0.3                36           64            0
0.4                42           52            6
0.5                 0            0          100
0.6                 0            0          100
0.7                 0            0          100
0.8                 0            0          100
0.9                 0          100            0
1.0                 0          100            0

Part médiane de coopérateurs dans les runs encore en coexistence à l'horizon :
T-slider
0.3    20.0
0.4    26.0
0.9    58.0
1.0    61.0

Durée médiane avant fixation (ticks) :
T-slider
0.1       9.0
0.2       7.0
0.3     717.0
0.4     513.5
0.5      50.5
0.6      46.0
0.7      52.0
0.8      48.0
0.9    2000.0
1.0    2000.0


## P4

In [22]:
"""
P4 : effet de la composition initiale x0.
Usage : python analyse_P4.py P4.csv

Chaque run est classé selon la même logique que P1 :
- coexistence  : coopérateurs et compétiteurs coexistent au moins SEUIL ticks
- coopération  : fixation en coopération avant SEUIL ticks
- compétition  : fixation en compétition avant SEUIL ticks
"""
import sys
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV = "./resultats/P4.csv"
SEUIL = 500           # même seuil que P1 (1/4 de l'horizon)
X_REF = 0.5           # valeur de référence

with open(CSV, encoding="utf-8") as f:
    skip = next(i for i, line in enumerate(f) if line.startswith('"[run number]"'))
df = pd.read_csv(CSV, skiprows=skip)

dove = df["dove-fixation?"].astype(str).str.lower() == "true"
df["regime"] = "Competition"
df.loc[dove, "regime"] = "Cooperation"
df.loc[df["[step]"] >= SEUIL, "regime"] = "Coexistence"

# Part des runs dans chaque régime, pour chaque valeur de R
parts = (pd.crosstab(df["initial-prop-dove-slider"], df["regime"], normalize="index")
           .reindex(columns=["Competition", "Coexistence", "Cooperation"], fill_value=0) * 100)
print(parts.round(0).astype(int).to_string())

# Figure : une courbe par régime
couleurs = {"Cooperation": "#e6a817", "Coexistence": "#2c7fb8", "Competition": "#3a9e8f"}
fig, ax = plt.subplots(figsize=(8, 4.5))
for regime, couleur in couleurs.items():
    ax.plot(parts.index, parts[regime], "o-", color=couleur, lw=2, ms=5, label=regime)
ax.axvline(X_REF, ls=":", color="0.4", lw=1)
ax.text(X_REF + 0.01, 95, "reference", fontsize=8, color="0.4")
ax.set_ylim(-3, 103)
ax.set_xlabel("Initial share of cooperators  $x_0$")
ax.set_ylabel("Share of runs (%)")
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
n = df.groupby("initial-prop-dove-slider").size().iloc[0]
ax.set_title("P4 — Regimes by initial composition\n"
             f"(simulation ; Z=100, N=5, $c_h$=$c_d$=0.8, T=0.4, R=1 ; {n} replications)",
             fontsize=10)
fig.tight_layout()
fig.savefig("./resultats/p4.png", dpi=300)

# Chiffres pour le texte
horizon = df["[step]"].max()
fin = df[df["[step]"] >= horizon].groupby("initial-prop-dove-slider")["prop-dove"].median()
print("\nPart médiane de coopérateurs dans les runs encore en coexistence à l'horizon :")
print((fin * 100).round(0).to_string())
print("\nDurée médiane avant fixation (ticks) :")
print(df.groupby("initial-prop-dove-slider")["[step]"].median().to_string())

regime                    Competition  Coexistence  Cooperation
initial-prop-dove-slider                                       
0.05                               40           60            0
0.10                               28           72            0
0.15                               36           62            2
0.20                               34           64            2
0.25                               34           66            0
0.30                               24           72            4
0.35                               34           66            0
0.40                               30           70            0
0.45                               30           68            2
0.50                               28           64            8
0.55                               40           50           10
0.60                               20           52           28
0.65                               18           48           34
0.70                               22   

## P5

In [25]:
"""
P5 : effet de la taille des groupes N (encombrement).
Usage : python analyse_P5.py P5.csv

Chaque run est classé selon la même logique que P1 :
- coexistence  : coopérateurs et compétiteurs coexistent au moins SEUIL ticks
- coopération  : fixation en coopération avant SEUIL ticks
- compétition  : fixation en compétition avant SEUIL ticks
"""
import sys
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CSV = "./resultats/P5.csv"
SEUIL = 500           # même seuil que P1 (1/4 de l'horizon)
N_REF = 5             # valeur de référence

with open(CSV, encoding="utf-8") as f:
    skip = next(i for i, line in enumerate(f) if line.startswith('"[run number]"'))
df = pd.read_csv(CSV, skiprows=skip)

dove = df["dove-fixation?"].astype(str).str.lower() == "true"
df["regime"] = "Competition"
df.loc[dove, "regime"] = "Cooperation"
df.loc[df["[step]"] >= SEUIL, "regime"] = "Coexistence"

# Part des runs dans chaque régime, pour chaque valeur de R
parts = (pd.crosstab(df["N-slider"], df["regime"], normalize="index")
           .reindex(columns=["Competition", "Coexistence", "Cooperation"], fill_value=0) * 100)
print(parts.round(0).astype(int).to_string())

# Figure : une courbe par régime
couleurs = {"Cooperation": "#e6a817", "Coexistence": "#2c7fb8", "Competition": "#3a9e8f"}
fig, ax = plt.subplots(figsize=(8, 4.5))
for regime, couleur in couleurs.items():
    ax.plot(parts.index, parts[regime], "o-", color=couleur, lw=2, ms=5, label=regime)
ax.axvline(N_REF, ls=":", color="0.4", lw=1)
ax.text(N_REF + 0.1, 95, "reference", fontsize=8, color="0.4")
ax.set_ylim(-3, 103)
ax.set_xlabel("Number of ventures per group  $N$")
ax.set_ylabel("Share of runs (%)")
ax.grid(alpha=0.25)
ax.legend(fontsize=9)
n = df.groupby("N-slider").size().iloc[0]
ax.set_title("P5 — Regimes by crowdedness\n"
             f"(simulation ; Z=100, $c_h$=$c_d$=0.8, T=0.4, R=1, $x_0$=0.5 ; {n} replications)",
             fontsize=10)
fig.tight_layout()
fig.savefig("./resultats/p5.png", dpi=300)

# Chiffres pour le texte
horizon = df["[step]"].max()
fin = df[df["[step]"] >= horizon].groupby("N-slider")["prop-dove"].median()
print("\nPart médiane de coopérateurs dans les runs encore en coexistence à l'horizon :")
print((fin * 100).round(0).to_string())
print("\nDurée médiane avant fixation (ticks) :")
print(df.groupby("N-slider")["[step]"].median().to_string())

regime    Competition  Coexistence  Cooperation
N-slider                                       
3                   0            0          100
4                   0            0          100
5                  26           66            8
6                   0            8           92
7                   0          100            0
8                   0           58           42
9                   0           98            2
10                  0          100            0
11                  0          100            0
12                  0          100            0
13                  0          100            0
14                  0          100            0
15                  0          100            0

Part médiane de coopérateurs dans les runs encore en coexistence à l'horizon :
N-slider
5     24.0
7     31.0
8     56.0
9     38.0
10    31.0
11    42.0
12    35.0
13    44.0
14    37.0
15    34.0

Durée médiane avant fixation (ticks) :
N-slider
3       37.5
4       62.5
5     